In [5]:
%pip install scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 3.0 MB/s  0:00:03m 3.1 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.3/35.3 MB 4.0 MB/s  0:00:08 eta 0:00:010:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6/6 [scikit-learn]0m 5/6 [scikit-learn]]
Note: you may need to restart the kernel to use updated packages.


In [1]:
from pathlib import Path
%pip install scikit-learn
# Rechercher la racine du projet en remontant depuis le dossier actuel
candidats = [Path.cwd(), *Path.cwd().parents]

RACINE_PROJET = next(
    (
        dossier for dossier in candidats
        if (dossier / "data").is_dir()
        and (dossier / "notebooks").is_dir()
    ),
    None
)

if RACINE_PROJET is None:
    raise FileNotFoundError("Impossible de trouver la racine du projet COFI.")

print("Racine du projet :", RACINE_PROJET)
print("Dossier des données brutes :", RACINE_PROJET / "data" / "raw")
print("Dossier des données traitées :", RACINE_PROJET / "data" / "processed")

Racine du projet : /home/nana-marc/Documents/coursBazongo/projetdeep/COFI
Dossier des données brutes : /home/nana-marc/Documents/coursBazongo/projetdeep/COFI/data/raw
Dossier des données traitées : /home/nana-marc/Documents/coursBazongo/projetdeep/COFI/data/processed


In [2]:
import pandas as pd

# Définir le chemin du corpus final
FICHIER_CORPUS = RACINE_PROJET / "data" / "processed" / "corpus_final.parquet"

# Charger le corpus
corpus = pd.read_parquet(FICHIER_CORPUS)

# Vérifier son contenu
print("Dimensions du corpus :", corpus.shape)

print("\nColonnes disponibles :")
print(corpus.columns.tolist())

print("\nRépartition des données par source :")
print(corpus["source"].value_counts())

print("\nValeurs manquantes :")
print(corpus[["incorrect", "correct", "source"]].isna().sum())

Dimensions du corpus : (67063, 5)

Colonnes disponibles :
['incorrect', 'correct', 'transformation', 'sec_transformation', 'source']

Répartition des données par source :
source
huggingface_multilingual_gec    61963
claude_5000                      5000
claude_100                        100
Name: count, dtype: int64

Valeurs manquantes :
incorrect    0
correct      0
source       0
dtype: int64


In [6]:
from sklearn.model_selection import train_test_split

# Première séparation : 90 % pour le développement, 10 % pour le test
train_val, test = train_test_split(
    corpus,
    test_size=0.10,
    random_state=42,
    stratify=corpus["source"]
)

# Deuxième séparation : 80 % pour l'entraînement, 10 % pour la validation
train, validation = train_test_split(
    train_val,
    test_size=1/9,
    random_state=42,
    stratify=train_val["source"]
)

# Réinitialiser les index
train = train.reset_index(drop=True)
validation = validation.reset_index(drop=True)
test = test.reset_index(drop=True)

print("Séparation terminée !")
print("Entraînement :", len(train))
print("Validation :", len(validation))
print("Test :", len(test))
print("Total :", len(train) + len(validation) + len(test))

Séparation terminée !
Entraînement : 53649
Validation : 6707
Test : 6707
Total : 67063


In [7]:
print("RÉPARTITION PAR SOURCE")

for nom, ensemble in [
    ("Entraînement", train),
    ("Validation", validation),
    ("Test", test)
]:
    print(f"\n{nom} :")
    print(ensemble["source"].value_counts())

    print("\nPourcentages :")
    print(
        (ensemble["source"].value_counts(normalize=True) * 100)
        .round(2)
    )

RÉPARTITION PAR SOURCE

Entraînement :
source
huggingface_multilingual_gec    49569
claude_5000                      4000
claude_100                         80
Name: count, dtype: int64

Pourcentages :
source
huggingface_multilingual_gec    92.40
claude_5000                      7.46
claude_100                       0.15
Name: proportion, dtype: float64

Validation :
source
huggingface_multilingual_gec    6197
claude_5000                      500
claude_100                        10
Name: count, dtype: int64

Pourcentages :
source
huggingface_multilingual_gec    92.40
claude_5000                      7.45
claude_100                       0.15
Name: proportion, dtype: float64

Test :
source
huggingface_multilingual_gec    6197
claude_5000                      500
claude_100                        10
Name: count, dtype: int64

Pourcentages :
source
huggingface_multilingual_gec    92.40
claude_5000                      7.45
claude_100                       0.15
Name: proportion, dtype: fl

In [8]:
def convertir_en_ensemble(df):
    return set(
        zip(df["incorrect"], df["correct"])
    )

paires_train = convertir_en_ensemble(train)
paires_validation = convertir_en_ensemble(validation)
paires_test = convertir_en_ensemble(test)

print("Chevauchements entre les ensembles :")

print(
    "Entraînement / Validation :",
    len(paires_train & paires_validation)
)

print(
    "Entraînement / Test :",
    len(paires_train & paires_test)
)

print(
    "Validation / Test :",
    len(paires_validation & paires_test)
)

Chevauchements entre les ensembles :
Entraînement / Validation : 0
Entraînement / Test : 0
Validation / Test : 0


In [9]:
# Définir le dossier de sauvegarde
DOSSIER_PROCESSED = RACINE_PROJET / "data" / "processed"
DOSSIER_PROCESSED.mkdir(parents=True, exist_ok=True)

# Regrouper les trois ensembles
ensembles = {
    "train": train,
    "validation": validation,
    "test": test
}

# Sauvegarder chaque ensemble en CSV et Parquet
for nom, ensemble in ensembles.items():

    fichier_csv = DOSSIER_PROCESSED / f"{nom}.csv"
    fichier_parquet = DOSSIER_PROCESSED / f"{nom}.parquet"

    ensemble.to_csv(
        fichier_csv,
        index=False,
        encoding="utf-8"
    )

    ensemble.to_parquet(
        fichier_parquet,
        index=False
    )

    print(f"{nom} sauvegardé :")
    print(f"  CSV : {fichier_csv}")
    print(f"  Parquet : {fichier_parquet}")

train sauvegardé :
  CSV : /home/nana-marc/Documents/coursBazongo/projetdeep/COFI/data/processed/train.csv
  Parquet : /home/nana-marc/Documents/coursBazongo/projetdeep/COFI/data/processed/train.parquet
validation sauvegardé :
  CSV : /home/nana-marc/Documents/coursBazongo/projetdeep/COFI/data/processed/validation.csv
  Parquet : /home/nana-marc/Documents/coursBazongo/projetdeep/COFI/data/processed/validation.parquet
test sauvegardé :
  CSV : /home/nana-marc/Documents/coursBazongo/projetdeep/COFI/data/processed/test.csv
  Parquet : /home/nana-marc/Documents/coursBazongo/projetdeep/COFI/data/processed/test.parquet


In [10]:
print("VÉRIFICATION DES FICHIERS SAUVEGARDÉS\n")

for nom in ["train", "validation", "test"]:

    fichier = DOSSIER_PROCESSED / f"{nom}.parquet"
    donnees = pd.read_parquet(fichier)

    print(f"{nom.upper()} :")
    print("Nombre de lignes :", len(donnees))
    print("Nombre de colonnes :", len(donnees.columns))
    print("Colonnes :", donnees.columns.tolist())
    print()

VÉRIFICATION DES FICHIERS SAUVEGARDÉS

TRAIN :
Nombre de lignes : 53649
Nombre de colonnes : 5
Colonnes : ['incorrect', 'correct', 'transformation', 'sec_transformation', 'source']

VALIDATION :
Nombre de lignes : 6707
Nombre de colonnes : 5
Colonnes : ['incorrect', 'correct', 'transformation', 'sec_transformation', 'source']

TEST :
Nombre de lignes : 6707
Nombre de colonnes : 5
Colonnes : ['incorrect', 'correct', 'transformation', 'sec_transformation', 'source']



In [11]:
# Calculer le nombre de mots dans chaque phrase
train["longueur_incorrecte"] = train["incorrect"].apply(
    lambda texte: len(texte.split())
)

train["longueur_correcte"] = train["correct"].apply(
    lambda texte: len(texte.split())
)

# Statistiques descriptives
print("LONGUEUR DES PHRASES INCORRECTES")
print(train["longueur_incorrecte"].describe(percentiles=[
    0.50, 0.75, 0.90, 0.95, 0.99
]))

print("\nLONGUEUR DES PHRASES CORRECTES")
print(train["longueur_correcte"].describe(percentiles=[
    0.50, 0.75, 0.90, 0.95, 0.99
]))

LONGUEUR DES PHRASES INCORRECTES
count    53649.000000
mean        14.236705
std          9.040314
min          1.000000
50%         12.000000
75%         20.000000
90%         28.000000
95%         33.000000
99%         39.000000
max         49.000000
Name: longueur_incorrecte, dtype: float64

LONGUEUR DES PHRASES CORRECTES
count    53649.000000
mean        14.292270
std          9.005033
min          1.000000
50%         12.000000
75%         20.000000
90%         28.000000
95%         33.000000
99%         39.000000
max         49.000000
Name: longueur_correcte, dtype: float64


In [12]:
seuil = 45

depassement_incorrect = (
    train["longueur_incorrecte"] > seuil
).sum()

depassement_correct = (
    train["longueur_correcte"] > seuil
).sum()

print("Phrases incorrectes dépassant 45 mots :",
      depassement_incorrect)

print("Phrases correctes dépassant 45 mots :",
      depassement_correct)

print("\nPourcentage de phrases incorrectes concernées :",
      round(depassement_incorrect / len(train) * 100, 3), "%")

print("Pourcentage de phrases correctes concernées :",
      round(depassement_correct / len(train) * 100, 3), "%")

Phrases incorrectes dépassant 45 mots : 35
Phrases correctes dépassant 45 mots : 36

Pourcentage de phrases incorrectes concernées : 0.065 %
Pourcentage de phrases correctes concernées : 0.067 %


Vérification de la longueur maximale

In [13]:
# Vérifier les phrases les plus longues

print("Les 5 phrases incorrectes les plus longues :")

print(
    train.nlargest(5, "longueur_incorrecte")[
        ["incorrect", "longueur_incorrecte"]
    ].to_string(index=False)
)

print("\nLes 5 phrases correctes les plus longues :")

print(
    train.nlargest(5, "longueur_correcte")[
        ["correct", "longueur_correcte"]
    ].to_string(index=False)
)

Les 5 phrases incorrectes les plus longues :
                                                                                                                                                                                                                                                      incorrect  longueur_incorrecte
          Un jour, le fils de la famille invite un ami de son ecole a dormir a la maison, quand soudain les deux garcons apercoivent un homme juste devant eux ; terrifie, son ami s'en va en courant de la maison et decide par la suite de couper le contact.                   49
          Un docteur rendit visite à un patient pour lui apporter une bonne et une mauvaise nouvelle : « La mauvaise nouvelle c'est que vous ne vivrez pas plus loin qu'à le terme du ce jour. Et voici la bonne nouvelle : j'ai oublié du vous le dire hier. »                   49
Un jours, le fils de la familles invite un amis de son écoles à dormir à la maisons, quand soudain les deux garçons aperçoiv

In [14]:
# Paramètres de longueur des séquences
LONGUEUR_MAX = 60

print("Longueur maximale définie :", LONGUEUR_MAX)

Longueur maximale définie : 60


In [15]:
import re
import unicodedata

def normaliser_texte(texte):
    texte = unicodedata.normalize("NFC", str(texte))
    texte = texte.strip()
    texte = re.sub(r"\s+", " ", texte)
    return texte


# Tester la fonction sur quelques exemples
exemples = [
    "  Il   etait fatigue.  ",
    "BONJOUR !",
    "J'aime   apprendre le français.",
    "  Nous   sommes   ici. "
]

for exemple in exemples:
    print("Avant :", repr(exemple))
    print("Après :", repr(normaliser_texte(exemple)))
    print("---")

Avant : '  Il   etait fatigue.  '
Après : 'Il etait fatigue.'
---
Avant : 'BONJOUR !'
Après : 'BONJOUR !'
---
Avant : "J'aime   apprendre le français."
Après : "J'aime apprendre le français."
---
Avant : '  Nous   sommes   ici. '
Après : 'Nous sommes ici.'
---


Appliquer la normalisation au corpus

In [16]:
# Normalisation des trois ensembles
for ensemble in [train, validation, test]:
    ensemble["incorrect"] = ensemble["incorrect"].apply(normaliser_texte)
    ensemble["correct"] = ensemble["correct"].apply(normaliser_texte)

print("Normalisation terminée.")

print("\nExemple de phrases normalisées :")
display(train[["incorrect", "correct"]].head(10))

Normalisation terminée.

Exemple de phrases normalisées :


,incorrect,correct
0,"L'effet de la morale, relativement aux passion...","L'effet de la morale, relativement aux passion..."
1,"Ainsi par exemple, les instructions de traitem...","Ainsi par exemple, les instructions de traitem..."
2,"Cependant depuis l'arrêts de l'original, la ve...","Cependant depuis l'arrêt de l'original, la ver..."
3,Il vient manger ce soir à le maison.,Il vient manger ce soir à la maison.
4,Si les transporteurs intestinaux favorisant l'...,Si les transporteurs intestinaux favorisant l'...
5,Le grand-mère tranquille ne lira pas votre jou...,La grand-mère tranquille ne lira pas votre jou...
6,Qu'sont-ce qui t'ont poussions à déménager ?,Qu'est-ce qui t'a poussé à déménager ?
7,L'utilisation de les planches et burladeros pe...,L'utilisation des planches et burladeros peint...
8,La boîte était pleine du livres.,La boîte était pleine de livres.
9,Il a lutté contre les discrimination raciale.,Il a lutté contre la discrimination raciale.


Tokenisation:
Nous allons maintenant découper les phrases en unités appelées tokens.

In [17]:
def tokeniser(texte):
    return re.findall(
        r"[^\W_]+(?:['’][^\W_]+)*|[^\w\s]",
        texte,
        flags=re.UNICODE
    )


# Quelques tests
exemples = [
    "Il vient manger ce soir à la maison.",
    "J'aime apprendre le français !",
    "L'école est-elle ouverte aujourd'hui ?",
    "Qu'est-ce que tu fais ?"
]

for phrase in exemples:
    print("Phrase :", phrase)
    print("Tokens :", tokeniser(phrase))
    print("---")

Phrase : Il vient manger ce soir à la maison.
Tokens : ['Il', 'vient', 'manger', 'ce', 'soir', 'à', 'la', 'maison', '.']
---
Phrase : J'aime apprendre le français !
Tokens : ["J'aime", 'apprendre', 'le', 'français', '!']
---
Phrase : L'école est-elle ouverte aujourd'hui ?
Tokens : ["L'école", 'est', '-', 'elle', 'ouverte', "aujourd'hui", '?']
---
Phrase : Qu'est-ce que tu fais ?
Tokens : ["Qu'est", '-', 'ce', 'que', 'tu', 'fais', '?']
---


In [18]:
# Tokenisation des phrases incorrectes et correctes

for ensemble in [train, validation, test]:
    ensemble["tokens_incorrect"] = ensemble["incorrect"].apply(tokeniser)
    ensemble["tokens_correct"] = ensemble["correct"].apply(tokeniser)

print("Tokenisation terminée !")

Tokenisation terminée !


In [19]:
# Afficher quelques exemples du corpus d'entraînement

display(
    train[
        ["incorrect", "tokens_incorrect", "correct", "tokens_correct"]
    ].head(10)
)

,incorrect,tokens_incorrect,correct,tokens_correct
0,"L'effet de la morale, relativement aux passion...","[L'effet, de, la, morale, ,, relativement, aux...","L'effet de la morale, relativement aux passion...","[L'effet, de, la, morale, ,, relativement, aux..."
1,"Ainsi par exemple, les instructions de traitem...","[Ainsi, par, exemple, ,, les, instructions, de...","Ainsi par exemple, les instructions de traitem...","[Ainsi, par, exemple, ,, les, instructions, de..."
2,"Cependant depuis l'arrêts de l'original, la ve...","[Cependant, depuis, l'arrêts, de, l'original, ...","Cependant depuis l'arrêt de l'original, la ver...","[Cependant, depuis, l'arrêt, de, l'original, ,..."
3,Il vient manger ce soir à le maison.,"[Il, vient, manger, ce, soir, à, le, maison, .]",Il vient manger ce soir à la maison.,"[Il, vient, manger, ce, soir, à, la, maison, .]"
4,Si les transporteurs intestinaux favorisant l'...,"[Si, les, transporteurs, intestinaux, favorisa...",Si les transporteurs intestinaux favorisant l'...,"[Si, les, transporteurs, intestinaux, favorisa..."
5,Le grand-mère tranquille ne lira pas votre jou...,"[Le, grand, -, mère, tranquille, ne, lira, pas...",La grand-mère tranquille ne lira pas votre jou...,"[La, grand, -, mère, tranquille, ne, lira, pas..."
6,Qu'sont-ce qui t'ont poussions à déménager ?,"[Qu'sont, -, ce, qui, t'ont, poussions, à, dém...",Qu'est-ce qui t'a poussé à déménager ?,"[Qu'est, -, ce, qui, t'a, poussé, à, déménager..."
7,L'utilisation de les planches et burladeros pe...,"[L'utilisation, de, les, planches, et, burlade...",L'utilisation des planches et burladeros peint...,"[L'utilisation, des, planches, et, burladeros,..."
8,La boîte était pleine du livres.,"[La, boîte, était, pleine, du, livres, .]",La boîte était pleine de livres.,"[La, boîte, était, pleine, de, livres, .]"
9,Il a lutté contre les discrimination raciale.,"[Il, a, lutté, contre, les, discrimination, ra...",Il a lutté contre la discrimination raciale.,"[Il, a, lutté, contre, la, discrimination, rac..."


In [20]:
# Rassembler tous les tokens des phrases incorrectes
tokens_incorrect_train = [
    token
    for phrase in train["tokens_incorrect"]
    for token in phrase
]

# Rassembler tous les tokens des phrases correctes
tokens_correct_train = [
    token
    for phrase in train["tokens_correct"]
    for token in phrase
]

# Combiner les deux listes
tous_les_tokens_train = (
    tokens_incorrect_train + tokens_correct_train
)

print("Nombre total de tokens :", len(tous_les_tokens_train))
print("Nombre de tokens différents :", len(set(tous_les_tokens_train)))

Nombre total de tokens : 1752942
Nombre de tokens différents : 38121


In [21]:
# Tokens spéciaux
PAD_TOKEN = "<PAD>"
UNK_TOKEN = "<UNK>"
SOS_TOKEN = "<SOS>"
EOS_TOKEN = "<EOS>"

tokens_speciaux = [
    PAD_TOKEN,
    UNK_TOKEN,
    SOS_TOKEN,
    EOS_TOKEN
]

print("Tokens spéciaux :", tokens_speciaux)

Tokens spéciaux : ['<PAD>', '<UNK>', '<SOS>', '<EOS>']


In [25]:
# Vérifier que les tokens spéciaux ne sont pas déjà dans le corpus
for token in tokens_speciaux:
    if token in set(tous_les_tokens_train):
        raise ValueError(f"Collision avec un token spécial : {token}")

# Construire la liste du vocabulaire
vocabulaire = tokens_speciaux + sorted(set(tous_les_tokens_train))

# Associer chaque token à un identifiant unique
token_vers_id = {
    token: identifiant
    for identifiant, token in enumerate(vocabulaire)
}

print("Nombre total d'entrées dans le vocabulaire :", len(vocabulaire))
print("\nQuelques exemples :")

for token in vocabulaire[:15]:
    print(token, "→", token_vers_id[token])

Nombre total d'entrées dans le vocabulaire : 38125

Quelques exemples :
<PAD> → 0
<UNK> → 1
<SOS> → 2
<EOS> → 3
! → 4
" → 5
$ → 6
% → 7
& → 8
' → 9
( → 10
) → 11
+ → 12
, → 13
- → 14


In [26]:
print("Identifiants des tokens spéciaux :")

for token in tokens_speciaux:
    print(f"{token} : {token_vers_id[token]}")

Identifiants des tokens spéciaux :
<PAD> : 0
<UNK> : 1
<SOS> : 2
<EOS> : 3


la transformation du langage naturel en représentation numérique.

In [27]:
def encoder_tokens(tokens, token_vers_id):
    return [
        token_vers_id.get(token, token_vers_id[UNK_TOKEN])
        for token in tokens
    ]

In [28]:
phrase = "Il vient manger."

tokens = tokeniser(phrase)

identifiants = encoder_tokens(tokens, token_vers_id)

print("Phrase :", phrase)
print("Tokens :", tokens)
print("Identifiants :", identifiants)

Phrase : Il vient manger.
Tokens : ['Il', 'vient', 'manger', '.']
Identifiants : [3322, 36953, 26274, 15]


Conversion des phrases en identifiants numériques

In [29]:
# Conversion des phrases incorrectes et correctes en identifiants

train["ids_incorrect"] = train["tokens_incorrect"].apply(
    lambda tokens: encoder_tokens(tokens, token_vers_id)
)

train["ids_correct"] = train["tokens_correct"].apply(
    lambda tokens: encoder_tokens(tokens, token_vers_id)
)

print("Conversion numérique de l'ensemble d'entraînement terminée.")

display(
    train[
        ["incorrect", "ids_incorrect", "correct", "ids_correct"]
    ].head(5)
)

Conversion numérique de l'ensemble d'entraînement terminée.


,incorrect,ids_incorrect,correct,ids_correct
0,"L'effet de la morale, relativement aux passion...","[4456, 14628, 24602, 27078, 13, 31856, 10253, ...","L'effet de la morale, relativement aux passion...","[4456, 14628, 24602, 27078, 13, 31856, 10253, ..."
1,"Ainsi par exemple, les instructions de traitem...","[678, 28842, 17711, 13, 24918, 20254, 14628, 3...","Ainsi par exemple, les instructions de traitem...","[678, 28842, 17711, 13, 24918, 20254, 14628, 3..."
2,"Cependant depuis l'arrêts de l'original, la ve...","[1551, 14903, 21824, 14628, 23517, 13, 24602, ...","Cependant depuis l'arrêt de l'original, la ver...","[1551, 14903, 21822, 14628, 23517, 13, 24602, ..."
3,Il vient manger ce soir à le maison.,"[3322, 36953, 26274, 11541, 34472, 37470, 2485...",Il vient manger ce soir à la maison.,"[3322, 36953, 26274, 11541, 34472, 37470, 2460..."
4,Si les transporteurs intestinaux favorisant l'...,"[7421, 24918, 36178, 20405, 18070, 21515, 3455...",Si les transporteurs intestinaux favorisant l'...,"[7421, 24918, 36178, 20405, 18070, 21515, 3455..."


In [30]:
# Conversion de l'ensemble de validation
validation["ids_incorrect"] = validation["tokens_incorrect"].apply(
    lambda tokens: encoder_tokens(tokens, token_vers_id)
)

validation["ids_correct"] = validation["tokens_correct"].apply(
    lambda tokens: encoder_tokens(tokens, token_vers_id)
)


# Conversion de l'ensemble de test
test["ids_incorrect"] = test["tokens_incorrect"].apply(
    lambda tokens: encoder_tokens(tokens, token_vers_id)
)

test["ids_correct"] = test["tokens_correct"].apply(
    lambda tokens: encoder_tokens(tokens, token_vers_id)
)

print("Conversion numérique terminée pour les trois ensembles.")

Conversion numérique terminée pour les trois ensembles.


In [31]:
for nom, ensemble in [
    ("Entraînement", train),
    ("Validation", validation),
    ("Test", test)
]:
    inconnus = sum(
        identifiant == token_vers_id[UNK_TOKEN]
        for colonne in ["ids_incorrect", "ids_correct"]
        for sequence in ensemble[colonne]
        for identifiant in sequence
    )

    print(f"{nom} : {inconnus} tokens inconnus")

Entraînement : 0 tokens inconnus
Validation : 1398 tokens inconnus
Test : 1322 tokens inconnus


In [32]:
for nom, ensemble in [
    ("Entraînement", train),
    ("Validation", validation),
    ("Test", test)
]:
    total = sum(
        len(sequence)
        for colonne in ["ids_incorrect", "ids_correct"]
        for sequence in ensemble[colonne]
    )

    inconnus = sum(
        identifiant == token_vers_id[UNK_TOKEN]
        for colonne in ["ids_incorrect", "ids_correct"]
        for sequence in ensemble[colonne]
        for identifiant in sequence
    )

    proportion = (inconnus / total) * 100

    print(f"\n{nom}")
    print("Total de tokens :", total)
    print("Tokens inconnus :", inconnus)
    print(f"Proportion inconnue : {proportion:.4f} %")


Entraînement
Total de tokens : 1752942
Tokens inconnus : 0
Proportion inconnue : 0.0000 %

Validation
Total de tokens : 219782
Tokens inconnus : 1398
Proportion inconnue : 0.6361 %

Test
Total de tokens : 218697
Tokens inconnus : 1322
Proportion inconnue : 0.6045 %


In [33]:
# Calculer les longueurs des séquences tokenisées
longueurs_incorrectes = train["ids_incorrect"].apply(len)
longueurs_correctes = train["ids_correct"].apply(len)

print("Longueur maximale des phrases incorrectes :",
      longueurs_incorrectes.max())

print("Longueur maximale des phrases correctes :",
      longueurs_correctes.max())

print("\nPhrases incorrectes dépassant 60 tokens :",
      (longueurs_incorrectes > 60).sum())

print("Phrases correctes dépassant 60 tokens :",
      (longueurs_correctes > 60).sum())

Longueur maximale des phrases incorrectes : 54
Longueur maximale des phrases correctes : 54

Phrases incorrectes dépassant 60 tokens : 0
Phrases correctes dépassant 60 tokens : 0


In [34]:
import numpy as np

LONGUEUR_MAX = 60

def appliquer_padding(sequence, longueur_max):
    sequence = sequence[:longueur_max]

    nombre_paddings = longueur_max - len(sequence)

    sequence = sequence + [token_vers_id[PAD_TOKEN]] * nombre_paddings

    return sequence

In [35]:
exemples = [
    [15, 28, 39],
    [15, 28, 39, 42, 56],
    list(range(1, 66))
]

for sequence in exemples:
    resultat = appliquer_padding(sequence, LONGUEUR_MAX)

    print("Longueur initiale :", len(sequence))
    print("Longueur finale :", len(resultat))
    print("Résultat :", resultat[:10], "...")
    print("---")

Longueur initiale : 3
Longueur finale : 60
Résultat : [15, 28, 39, 0, 0, 0, 0, 0, 0, 0] ...
---
Longueur initiale : 5
Longueur finale : 60
Résultat : [15, 28, 39, 42, 56, 0, 0, 0, 0, 0] ...
---
Longueur initiale : 65
Longueur finale : 60
Résultat : [1, 2, 3, 4, 5, 6, 7, 8, 9, 10] ...
---


In [36]:
# Padding des phrases incorrectes

for ensemble in [train, validation, test]:
    ensemble["ids_incorrect_pad"] = ensemble["ids_incorrect"].apply(
        lambda sequence: appliquer_padding(sequence, LONGUEUR_MAX)
    )

print("Padding des phrases incorrectes terminé.")

Padding des phrases incorrectes terminé.


In [37]:
for nom, ensemble in [
    ("Entraînement", train),
    ("Validation", validation),
    ("Test", test)
]:
    longueurs = ensemble["ids_incorrect_pad"].apply(len)

    print(f"\n{nom}")
    print("Nombre de phrases :", len(ensemble))
    print("Longueur minimale :", longueurs.min())
    print("Longueur maximale :", longueurs.max())


Entraînement
Nombre de phrases : 53649
Longueur minimale : 60
Longueur maximale : 60

Validation
Nombre de phrases : 6707
Longueur minimale : 60
Longueur maximale : 60

Test
Nombre de phrases : 6707
Longueur minimale : 60
Longueur maximale : 60


In [38]:
LONGUEUR_MAX_CIBLE = LONGUEUR_MAX + 2

print("Longueur maximale de la phrase :", LONGUEUR_MAX)
print("Longueur maximale de la cible avec SOS et EOS :", LONGUEUR_MAX_CIBLE)

Longueur maximale de la phrase : 60
Longueur maximale de la cible avec SOS et EOS : 62


In [40]:
LONGUEUR_MAX_CIBLE = LONGUEUR_MAX + 2

def preparer_cible(sequence):
    # Ajouter SOS au début et EOS à la fin
    sequence = (
        [token_vers_id[SOS_TOKEN]]
        + sequence[:LONGUEUR_MAX]
        + [token_vers_id[EOS_TOKEN]]
    )

    # Compléter avec PAD jusqu'à 62 tokens
    nombre_paddings = LONGUEUR_MAX_CIBLE - len(sequence)
    sequence = sequence + [token_vers_id[PAD_TOKEN]] * nombre_paddings

    return sequence

In [41]:
for ensemble in [train, validation, test]:
    ensemble["cibles_completes"] = ensemble["ids_correct"].apply(
        preparer_cible
    )

print("Préparation des cibles terminée.")

Préparation des cibles terminée.


In [42]:
for nom, ensemble in [
    ("Entraînement", train),
    ("Validation", validation),
    ("Test", test)
]:
    longueurs = ensemble["cibles_completes"].apply(len)

    print(f"\n{nom}")
    print("Nombre de phrases :", len(ensemble))
    print("Longueur minimale :", longueurs.min())
    print("Longueur maximale :", longueurs.max())


Entraînement
Nombre de phrases : 53649
Longueur minimale : 62
Longueur maximale : 62

Validation
Nombre de phrases : 6707
Longueur minimale : 62
Longueur maximale : 62

Test
Nombre de phrases : 6707
Longueur minimale : 62
Longueur maximale : 62


Construire les entrées du décodeur

In [43]:
def preparer_entree_decodeur(sequence):
    return sequence[:-1]


for ensemble in [train, validation, test]:
    ensemble["decoder_input"] = ensemble["cibles_completes"].apply(
        preparer_entree_decodeur
    )

print("Entrées du décodeur préparées.")

Entrées du décodeur préparées.


Construire les sorties attendues du décodeur

In [44]:
def preparer_sortie_decodeur(sequence):
    return sequence[1:]


for ensemble in [train, validation, test]:
    ensemble["decoder_target"] = ensemble["cibles_completes"].apply(
        preparer_sortie_decodeur
    )

print("Sorties du décodeur préparées.")

Sorties du décodeur préparées.


In [45]:
for nom, ensemble in [
    ("Entraînement", train),
    ("Validation", validation),
    ("Test", test)
]:
    longueurs_entree = ensemble["decoder_input"].apply(len)
    longueurs_sortie = ensemble["decoder_target"].apply(len)

    print(f"\n{nom}")
    print("Entrées du décodeur :", longueurs_entree.min(),
          "-", longueurs_entree.max())
    print("Sorties du décodeur :", longueurs_sortie.min(),
          "-", longueurs_sortie.max())


Entraînement
Entrées du décodeur : 61 - 61
Sorties du décodeur : 61 - 61

Validation
Entrées du décodeur : 61 - 61
Sorties du décodeur : 61 - 61

Test
Entrées du décodeur : 61 - 61
Sorties du décodeur : 61 - 61


In [46]:
# Choisir une phrase de l'ensemble d'entraînement
indice = 0

print("Phrase incorrecte :")
print(train.loc[indice, "incorrect"])

print("\nPhrase correcte :")
print(train.loc[indice, "correct"])

print("\nCible complète (IDs) :")
print(train.loc[indice, "cibles_completes"])

print("\nEntrée du décodeur (IDs) :")
print(train.loc[indice, "decoder_input"])

print("\nSortie attendue du décodeur (IDs) :")
print(train.loc[indice, "decoder_target"])

Phrase incorrecte :
L'effet de la morale, relativement aux passions est d'aider les hommes en être hommes.

Phrase correcte :
L'effet de la morale, relativement aux passions est d'aider les hommes à être hommes.

Cible complète (IDs) :
[2, 4456, 14628, 24602, 27078, 13, 31856, 10253, 29125, 17523, 13898, 24918, 19529, 37470, 38048, 19529, 15, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]

Entrée du décodeur (IDs) :
[2, 4456, 14628, 24602, 27078, 13, 31856, 10253, 29125, 17523, 13898, 24918, 19529, 37470, 38048, 19529, 15, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]

Sortie attendue du décodeur (IDs) :
[4456, 14628, 24602, 27078, 13, 31856, 10253, 29125, 17523, 13898, 24918, 19529, 37470, 38048, 19529, 15, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 

In [47]:
for nom, ensemble in [
    ("Entraînement", train),
    ("Validation", validation),
    ("Test", test)
]:
    print(f"\n========== {nom} ==========")

    print("Nombre de phrases :", len(ensemble))

    print("Encodeur :", ensemble["ids_incorrect_pad"].apply(len).unique())

    print("Entrée décodeur :", ensemble["decoder_input"].apply(len).unique())

    print("Sortie décodeur :", ensemble["decoder_target"].apply(len).unique())

    print("Cible complète :", ensemble["cibles_completes"].apply(len).unique())


========== Entraînement ==========
Nombre de phrases : 53649
Encodeur : [60]
Entrée décodeur : [61]
Sortie décodeur : [61]
Cible complète : [62]

========== Validation ==========
Nombre de phrases : 6707
Encodeur : [60]
Entrée décodeur : [61]
Sortie décodeur : [61]
Cible complète : [62]

========== Test ==========
Nombre de phrases : 6707
Encodeur : [60]
Entrée décodeur : [61]
Sortie décodeur : [61]
Cible complète : [62]


In [48]:
for nom, ensemble in [
    ("Entraînement", train),
    ("Validation", validation),
    ("Test", test)
]:
    print(f"\n========== {nom} ==========")

    # Vérifier le début de la cible complète
    sos_correct = ensemble["cibles_completes"].apply(
        lambda seq: seq[0] == token_vers_id[SOS_TOKEN]
    ).all()

    # Vérifier la présence de EOS dans les cibles
    eos_present = ensemble["cibles_completes"].apply(
        lambda seq: token_vers_id[EOS_TOKEN] in seq
    ).all()

    # Vérifier que l'encodeur commence par un vrai token
    pad_debut = ensemble["ids_incorrect_pad"].apply(
        lambda seq: seq[0] == token_vers_id[PAD_TOKEN]
    ).sum()

    print("Toutes les cibles commencent par SOS :", sos_correct)
    print("Toutes les cibles contiennent EOS :", eos_present)
    print("Phrases de l'encodeur commençant par PAD :", pad_debut)


========== Entraînement ==========
Toutes les cibles commencent par SOS : True
Toutes les cibles contiennent EOS : True
Phrases de l'encodeur commençant par PAD : 0

========== Validation ==========
Toutes les cibles commencent par SOS : True
Toutes les cibles contiennent EOS : True
Phrases de l'encodeur commençant par PAD : 0

========== Test ==========
Toutes les cibles commencent par SOS : True
Toutes les cibles contiennent EOS : True
Phrases de l'encodeur commençant par PAD : 0


In [49]:
def decoder_ids(identifiants, id_vers_token):
    return [
        id_vers_token.get(identifiant, UNK_TOKEN)
        for identifiant in identifiants
    ]


# Construire le dictionnaire inverse
id_vers_token = {
    identifiant: token
    for token, identifiant in token_vers_id.items()
}

indice = 0

print("Phrase incorrecte originale :")
print(train.loc[indice, "incorrect"])

print("\nTokens reconstruits depuis l'encodeur :")
print(decoder_ids(train.loc[indice, "ids_incorrect"], id_vers_token))

print("\nPhrase correcte originale :")
print(train.loc[indice, "correct"])

print("\nTokens reconstruits depuis la cible :")
print(decoder_ids(
    train.loc[indice, "cibles_completes"],
    id_vers_token
))

Phrase incorrecte originale :
L'effet de la morale, relativement aux passions est d'aider les hommes en être hommes.

Tokens reconstruits depuis l'encodeur :
["L'effet", 'de', 'la', 'morale', ',', 'relativement', 'aux', 'passions', 'est', "d'aider", 'les', 'hommes', 'en', 'être', 'hommes', '.']

Phrase correcte originale :
L'effet de la morale, relativement aux passions est d'aider les hommes à être hommes.

Tokens reconstruits depuis la cible :
['<SOS>', "L'effet", 'de', 'la', 'morale', ',', 'relativement', 'aux', 'passions', 'est', "d'aider", 'les', 'hommes', 'à', 'être', 'hommes', '.', '<EOS>', '<PAD>', '<PAD>', '<PAD>', '<PAD>', '<PAD>', '<PAD>', '<PAD>', '<PAD>', '<PAD>', '<PAD>', '<PAD>', '<PAD>', '<PAD>', '<PAD>', '<PAD>', '<PAD>', '<PAD>', '<PAD>', '<PAD>', '<PAD>', '<PAD>', '<PAD>', '<PAD>', '<PAD>', '<PAD>', '<PAD>', '<PAD>', '<PAD>', '<PAD>', '<PAD>', '<PAD>', '<PAD>', '<PAD>', '<PAD>', '<PAD>', '<PAD>', '<PAD>', '<PAD>', '<PAD>', '<PAD>', '<PAD>', '<PAD>', '<PAD>', '<PAD>']

In [50]:
import numpy as np

def convertir_en_tableaux(ensemble):
    X_encodeur = np.array(
        ensemble["ids_incorrect_pad"].tolist(),
        dtype=np.int32
    )

    X_decodeur = np.array(
        ensemble["decoder_input"].tolist(),
        dtype=np.int32
    )

    Y_decodeur = np.array(
        ensemble["decoder_target"].tolist(),
        dtype=np.int32
    )

    return X_encodeur, X_decodeur, Y_decodeur


X_train, decoder_train, Y_train = convertir_en_tableaux(train)

X_val, decoder_val, Y_val = convertir_en_tableaux(validation)

X_test, decoder_test, Y_test = convertir_en_tableaux(test)

print("Conversion en tableaux NumPy terminée.")

Conversion en tableaux NumPy terminée.


In [51]:
for nom, X, D, Y in [
    ("Entraînement", X_train, decoder_train, Y_train),
    ("Validation", X_val, decoder_val, Y_val),
    ("Test", X_test, decoder_test, Y_test)
]:
    print(f"\n========== {nom} ==========")
    print("Encodeur :", X.shape)
    print("Entrée décodeur :", D.shape)
    print("Sortie décodeur :", Y.shape)


========== Entraînement ==========
Encodeur : (53649, 60)
Entrée décodeur : (53649, 61)
Sortie décodeur : (53649, 61)

========== Validation ==========
Encodeur : (6707, 60)
Entrée décodeur : (6707, 61)
Sortie décodeur : (6707, 61)

========== Test ==========
Encodeur : (6707, 60)
Entrée décodeur : (6707, 61)
Sortie décodeur : (6707, 61)


In [52]:
for nom, X, D, Y in [
    ("Entraînement", X_train, decoder_train, Y_train),
    ("Validation", X_val, decoder_val, Y_val),
    ("Test", X_test, decoder_test, Y_test)
]:
    print(f"\n========== {nom} ==========")
    print("Type encodeur :", X.dtype)
    print("Type entrée décodeur :", D.dtype)
    print("Type sortie décodeur :", Y.dtype)


========== Entraînement ==========
Type encodeur : int32
Type entrée décodeur : int32
Type sortie décodeur : int32

========== Validation ==========
Type encodeur : int32
Type entrée décodeur : int32
Type sortie décodeur : int32

========== Test ==========
Type encodeur : int32
Type entrée décodeur : int32
Type sortie décodeur : int32


In [53]:
import numpy as np

DOSSIER_PROCESSED = RACINE_PROJET / "data" / "processed"
DOSSIER_PROCESSED.mkdir(parents=True, exist_ok=True)

# Entraînement
np.savez_compressed(
    DOSSIER_PROCESSED / "train_prepared.npz",
    encoder=X_train,
    decoder_input=decoder_train,
    decoder_target=Y_train
)

# Validation
np.savez_compressed(
    DOSSIER_PROCESSED / "validation_prepared.npz",
    encoder=X_val,
    decoder_input=decoder_val,
    decoder_target=Y_val
)

# Test
np.savez_compressed(
    DOSSIER_PROCESSED / "test_prepared.npz",
    encoder=X_test,
    decoder_input=decoder_test,
    decoder_target=Y_test
)

print("Les trois ensembles ont été sauvegardés.")

Les trois ensembles ont été sauvegardés.


In [54]:
import json

FICHIER_VOCABULAIRE = DOSSIER_PROCESSED / "vocabulaire.json"

with open(FICHIER_VOCABULAIRE, "w", encoding="utf-8") as fichier:
    json.dump(
        token_vers_id,
        fichier,
        ensure_ascii=False,
        indent=4
    )

print("Vocabulaire sauvegardé :", FICHIER_VOCABULAIRE)
print("Nombre de tokens :", len(vocabulaire))

Vocabulaire sauvegardé : /home/nana-marc/Documents/coursBazongo/projetdeep/COFI/data/processed/vocabulaire.json
Nombre de tokens : 38125


In [55]:
parametres_preparation = {
    "longueur_max_encodeur": LONGUEUR_MAX,
    "longueur_max_cible_complete": LONGUEUR_MAX_CIBLE,
    "longueur_decodeur": LONGUEUR_MAX_CIBLE - 1,
    "taille_vocabulaire": len(vocabulaire),
    "tokens_speciaux": {
        "PAD": {
            "token": PAD_TOKEN,
            "id": token_vers_id[PAD_TOKEN]
        },
        "UNK": {
            "token": UNK_TOKEN,
            "id": token_vers_id[UNK_TOKEN]
        },
        "SOS": {
            "token": SOS_TOKEN,
            "id": token_vers_id[SOS_TOKEN]
        },
        "EOS": {
            "token": EOS_TOKEN,
            "id": token_vers_id[EOS_TOKEN]
        }
    }
}

FICHIER_PARAMETRES = DOSSIER_PROCESSED / "parametres_preparation.json"

with open(FICHIER_PARAMETRES, "w", encoding="utf-8") as fichier:
    json.dump(
        parametres_preparation,
        fichier,
        ensure_ascii=False,
        indent=4
    )

print("Paramètres sauvegardés :", FICHIER_PARAMETRES)

Paramètres sauvegardés : /home/nana-marc/Documents/coursBazongo/projetdeep/COFI/data/processed/parametres_preparation.json


In [56]:
# Recharger les fichiers NumPy
train_charge = np.load(DOSSIER_PROCESSED / "train_prepared.npz")
validation_charge = np.load(DOSSIER_PROCESSED / "validation_prepared.npz")
test_charge = np.load(DOSSIER_PROCESSED / "test_prepared.npz")

print("Fichiers NumPy rechargés avec succès.")

Fichiers NumPy rechargés avec succès.


In [57]:
for nom, ensemble in [
    ("Entraînement", train_charge),
    ("Validation", validation_charge),
    ("Test", test_charge)
]:
    print(f"\n========== {nom} ==========")
    print("Encodeur :", ensemble["encoder"].shape)
    print("Entrée décodeur :", ensemble["decoder_input"].shape)
    print("Sortie décodeur :", ensemble["decoder_target"].shape)


========== Entraînement ==========
Encodeur : (53649, 60)
Entrée décodeur : (53649, 61)
Sortie décodeur : (53649, 61)

========== Validation ==========
Encodeur : (6707, 60)
Entrée décodeur : (6707, 61)
Sortie décodeur : (6707, 61)

========== Test ==========
Encodeur : (6707, 60)
Entrée décodeur : (6707, 61)
Sortie décodeur : (6707, 61)


In [58]:
# Recharger le vocabulaire
with open(DOSSIER_PROCESSED / "vocabulaire.json", "r", encoding="utf-8") as fichier:
    vocabulaire_charge = json.load(fichier)

# Recharger les paramètres
with open(DOSSIER_PROCESSED / "parametres_preparation.json", "r", encoding="utf-8") as fichier:
    parametres_charges = json.load(fichier)

print("Taille du vocabulaire :", len(vocabulaire_charge))

print("\nTokens spéciaux :")
print(parametres_charges["tokens_speciaux"])

print("\nParamètres :")
print("Longueur maximale encodeur :", parametres_charges["longueur_max_encodeur"])
print("Longueur maximale cible complète :", parametres_charges["longueur_max_cible_complete"])
print("Longueur décodeur :", parametres_charges["longueur_decodeur"])
print("Taille du vocabulaire :", parametres_charges["taille_vocabulaire"])


Taille du vocabulaire : 38125

Tokens spéciaux :
{'PAD': {'token': '<PAD>', 'id': 0}, 'UNK': {'token': '<UNK>', 'id': 1}, 'SOS': {'token': '<SOS>', 'id': 2}, 'EOS': {'token': '<EOS>', 'id': 3}}

Paramètres :
Longueur maximale encodeur : 60
Longueur maximale cible complète : 62
Longueur décodeur : 61
Taille du vocabulaire : 38125
